# 02. SFT: Qwen3-8B + Unsloth + TRL

- ベースモデル: `unsloth/Qwen3-8B` (4bit QLoRA)
- データ: `YANS-official/ogiri-bokete` の `text_to_text` (約100お題 / 898回答)
- ハード想定: A100 40GB 単機 (Colab)
- 学習対象: assistant 応答のみ (`train_on_responses_only`)

## Colab セットアップ
ローカルで実行する場合は次のセルはスキップ可。

In [ ]:
import sys, os
IS_COLAB = "google.colab" in sys.modules
print("Colab:", IS_COLAB)

if IS_COLAB:
    REPO_URL = "https://github.com/watamoo/oogiri-AI.git"
    REPO_DIR = "/content/oogiri-AI"
    if os.path.exists(REPO_DIR):
        # 既に clone 済みなら pull で最新化 (毎回最新コードで実行する)
        get_ipython().system(f"cd {REPO_DIR} && git pull --ff-only")
    else:
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", f"{REPO_DIR}/notebooks")

    # 依存インストール
    get_ipython().run_line_magic("pip", "install -q --upgrade pip")
    get_ipython().run_line_magic("pip", 'install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
    get_ipython().run_line_magic("pip", 'install -q --no-deps "trl<0.20.0" peft accelerate bitsandbytes')
    get_ipython().run_line_magic("pip", "install -q datasets pandas pyarrow japanize-matplotlib")

In [ ]:
from pathlib import Path
import sys

# autoreload はローカル開発の利便性用。Colabの古いIPythonだと imp が無くてコケるのでskip
if "google.colab" not in sys.modules:
    try:
        get_ipython().run_line_magic("load_ext", "autoreload")
        get_ipython().run_line_magic("autoreload", "2")
    except Exception as e:
        print("autoreload skipped:", e)

# プロジェクトルートをsys.pathに追加し、src.* をimport可能にする
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CHECKPOINT_DIR = ROOT / "checkpoints" / "sft_qwen3_8b"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ROOT

## モデル読み込み

Qwen3-8B を 4bit でロード → LoRA を載せる。max_seq_length は大喜利が短文なので 512 で十分。

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 512
LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # 自動判定 (A100ならbf16)
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK * 2,  # alpha=2r で学習が速い (Unsloth推奨)
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## データ準備

In [ ]:
from src.data import load_oogiri_t2t, to_chat_dataset, SYSTEM_PROMPT

# parquet が無ければ HF から自動生成 (src/data.py 内で吸収)
# 過剰なノイズを避けるため、各お題で score 上位5件に絞る
train_df, eval_df = load_oogiri_t2t(eval_ratio=0.1, seed=42, top_k=5)
print("train:", len(train_df), "eval:", len(eval_df))
display(train_df.head())

train_ds = to_chat_dataset(train_df)
eval_ds = to_chat_dataset(eval_df)

In [ ]:
# Qwen3 のチャットテンプレートを当てて text 化
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}

train_ds = train_ds.map(formatting_prompts_func, batched=True, remove_columns=["messages"])
eval_ds = eval_ds.map(formatting_prompts_func, batched=True, remove_columns=["messages"])

# テンプレ適用後のサンプル確認
print(train_ds[0]["text"][:500])

## SFTTrainer 設定

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        per_device_eval_batch_size=8,
        warmup_ratio=0.1,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        seed=3407,
        output_dir=str(CHECKPOINT_DIR),
        report_to="none",
    ),
)

# user 部分を loss から外し、assistant 応答のみで学習する
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# 学習実行前のVRAM確認
gpu = torch.cuda.get_device_properties(0)
used = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total = round(gpu.total_memory / 1024**3, 2)
print("GPU:", gpu.name, "|", used, "/", total, "GB used")

## 学習

In [ ]:
trainer_stats = trainer.train()
trainer_stats

In [ ]:
# 学習後VRAMピーク確認
peak = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print("Peak VRAM:", peak, "GB")

## 学習後の生成サンプル

validation のお題（学習に使っていない）でモデル出力を確認する。データ中の高score回答も並べて比較できるようにする。

In [ ]:
FastLanguageModel.for_inference(model)

# validation のユニークお題を取得
eval_odai_df = (
    eval_df.drop_duplicates(subset=["odai_id"])[["odai_id", "odai"]]
    .reset_index(drop=True)
)
print("validationお題数:", len(eval_odai_df))

# 各お題で何回生成するか (多様性確認のため複数)
N_SAMPLES_PER_ODAI = 3

for _, row in eval_odai_df.iterrows():
    odai = row["odai"]
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": odai},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    # 学習データ中の参考回答 (高score上位3件)
    refs = (
        eval_df[eval_df["odai_id"] == row["odai_id"]]
        .sort_values("score", ascending=False)
        .head(3)[["text", "score"]]
    )

    print("=" * 60)
    print("お題:", odai)
    print("-- 参考(データ中の高score回答) --")
    for _, r in refs.iterrows():
        print(f"  [score={r['score']}] {r['text']}")
    print("-- モデル生成 --")
    for i in range(N_SAMPLES_PER_ODAI):
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=64,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
        )
        gen = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
        print(f"  [{i+1}] {gen.strip()}")
    print()

## LoRA アダプタ保存

In [ ]:
save_path = CHECKPOINT_DIR / "adapter"
model.save_pretrained(str(save_path))
tokenizer.save_pretrained(str(save_path))
print("saved adapter:", save_path)